# Daily Challenge: Multi-Attention and Transformer Comparisons

This notebook explores attention inside Transformer architectures. We implement single-head attention, extend it to multi-head attention, build a lightweight encoder-only classifier for Natural Language Inference, and compare it with a pretrained DistilBERT baseline.

## 0. Setup

The notebook is designed for Google Colab or a local Jupyter environment. Training cells use small sample limits so they can run quickly during experimentation.

In [ ]:
# Install dependencies if needed
!pip install -q torch transformers scikit-learn pandas matplotlib seaborn requests

In [ ]:
import io
import math
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Attention Concepts

| Type | What it does | Typical use |
|---|---|---|
| Single-head self-attention | Uses one attention pattern to let each token attend to all tokens in the same sequence. | A simple attention block for learning contextual token representations. |
| Multi-head self-attention | Runs several attention heads in parallel, then concatenates the results. Each head can focus on different relationships. | Core component of Transformer encoder and decoder blocks. |
| Cross-attention | Queries come from one sequence, while keys and values come from another sequence. | Encoder-decoder models, translation, summarization, retrieval-augmented generation. |

Single-head attention is easy to inspect but limited because it compresses all relationships into one attention map. Multi-head attention is more expressive: one head may focus on syntax, another on long-distance dependencies, and another on semantic similarity. Cross-attention is different because it connects two sources of information, for example a decoder attending to encoder outputs during translation.

## 2. Single-Head Attention Implementation

Scaled dot-product attention uses linear projections for queries, keys, and values. The attention scores are computed as `Q @ K.T / sqrt(d_k)`, normalized with softmax, then used to create a weighted sum of the values.

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.hidden_dim)
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask[:, None, :] == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context = torch.matmul(attention_weights, v)
        return context, attention_weights


# Shape validation with dummy tensors
batch_size, seq_len, hidden_dim = 2, 5, 16
dummy_x = torch.randn(batch_size, seq_len, hidden_dim)
dummy_mask = torch.ones(batch_size, seq_len)

single_attention = SingleHeadAttention(hidden_dim)
context, weights = single_attention(dummy_x, dummy_mask)

print("Input shape      :", dummy_x.shape)
print("Context shape    :", context.shape)
print("Attention shape  :", weights.shape)
print("Sample weights for batch 0, token 0:")
print(weights[0, 0].detach())

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(weights[0].detach().numpy(), cmap="viridis", annot=True, fmt=".2f")
plt.title("Single-head attention weights - dummy sample")
plt.xlabel("Key token position")
plt.ylabel("Query token position")
plt.show()

## 3. Multi-Head Attention Module

The multi-head module splits the hidden dimension into several smaller heads, applies attention independently in each head, concatenates the head outputs, and projects them back to the original hidden dimension. Dropout and residual connections are included to make the block closer to a Transformer-style component.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        x = x.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        batch_size, num_heads, seq_len, head_dim = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.reshape(batch_size, seq_len, num_heads * head_dim)

    def forward(self, x, attention_mask=None, return_attention=False):
        residual = x

        q = self.split_heads(self.q_proj(x))
        k = self.split_heads(self.k_proj(x))
        v = self.split_heads(self.v_proj(x))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask[:, None, None, :] == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context = torch.matmul(attention_weights, v)
        context = self.combine_heads(context)
        output = self.out_proj(context)
        output = residual + self.dropout(output)

        if return_attention:
            return output, attention_weights
        return output


multi_attention = MultiHeadAttention(hidden_dim=32, num_heads=4)
dummy_x = torch.randn(2, 7, 32)
dummy_mask = torch.ones(2, 7)
multi_context, multi_weights = multi_attention(dummy_x, dummy_mask, return_attention=True)

print("Input shape     :", dummy_x.shape)
print("Output shape    :", multi_context.shape)
print("Attention shape :", multi_weights.shape)  # batch, heads, query_len, key_len

## 4. Dataset Loading

We use the Natural Language Inference dataset ZIP provided for the bootcamp. The loader below is defensive: it extracts the ZIP, searches for tabular files, detects premise/hypothesis/label columns, and falls back to a tiny toy dataset if the remote file is unavailable during a live session.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/devtlv/Datasets-GEN-AI-Bootcamp/main/Week%206/W6D1%20GenAi%20France/Basics%20of%20BERT%20and%20XLM-RoBERTa%20-%20PyTorch%20-%202.zip"
DATA_DIR = Path("data/nli_attention")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def download_and_extract_dataset(url=DATA_URL, data_dir=DATA_DIR):
    zip_path = data_dir / "nli_dataset.zip"
    if not zip_path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        zip_path.write_bytes(response.content)

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(data_dir)

    return data_dir

def read_table(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".tsv", ".txt"]:
        return pd.read_csv(path, sep="\t")
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    return None

def find_nli_columns(df):
    lower_to_original = {col.lower(): col for col in df.columns}
    premise_names = ["premise", "sentence1", "sent1", "text_a", "question1"]
    hypothesis_names = ["hypothesis", "sentence2", "sent2", "text_b", "question2"]
    label_names = ["label", "labels", "gold_label", "target", "class"]

    premise_col = next((lower_to_original[name] for name in premise_names if name in lower_to_original), None)
    hypothesis_col = next((lower_to_original[name] for name in hypothesis_names if name in lower_to_original), None)
    label_col = next((lower_to_original[name] for name in label_names if name in lower_to_original), None)
    return premise_col, hypothesis_col, label_col

def load_nli_dataframe():
    try:
        root = download_and_extract_dataset()
        candidate_paths = []
        for pattern in ["*.csv", "*.tsv", "*.json", "*.jsonl", "*.txt"]:
            candidate_paths.extend(root.rglob(pattern))

        for path in candidate_paths:
            try:
                df = read_table(path)
                if df is None or df.empty:
                    continue
                premise_col, hypothesis_col, label_col = find_nli_columns(df)
                if premise_col and hypothesis_col and label_col:
                    print("Using dataset file:", path)
                    nli_df = df[[premise_col, hypothesis_col, label_col]].copy()
                    nli_df.columns = ["premise", "hypothesis", "label"]
                    return nli_df.dropna()
            except Exception as error:
                print("Skipping", path, "because", error)

        raise FileNotFoundError("No compatible NLI table found in the ZIP file.")

    except Exception as error:
        print("Dataset download/loading failed, using a tiny fallback dataset.")
        print("Reason:", error)
        return pd.DataFrame({
            "premise": [
                "A woman is playing a violin on stage.",
                "Two children are running through a park.",
                "A man is sleeping on the sofa.",
                "The chef is cutting vegetables in the kitchen.",
                "People are sitting inside a quiet library.",
                "A dog is swimming in a lake."
            ],
            "hypothesis": [
                "A person is performing music.",
                "The children are indoors.",
                "Someone is resting.",
                "The chef is preparing food.",
                "The library is crowded with dancers.",
                "An animal is in the water."
            ],
            "label": ["entailment", "contradiction", "entailment", "entailment", "contradiction", "entailment"]
        })

raw_df = load_nli_dataframe()
raw_df.head()

In [ ]:
def normalize_labels(df):
    df = df.copy()
    canonical = {"entailment": 0, "neutral": 1, "contradiction": 2}

    if pd.api.types.is_numeric_dtype(df["label"]):
        df["label_id"] = df["label"].astype(int)
        id_to_label = {0: "entailment", 1: "neutral", 2: "contradiction"}
    else:
        cleaned = df["label"].astype(str).str.lower().str.strip()
        df = df[cleaned.isin(canonical.keys())].copy()
        df["label_id"] = cleaned[cleaned.isin(canonical.keys())].map(canonical).astype(int)
        id_to_label = {value: key for key, value in canonical.items()}

    df = df[["premise", "hypothesis", "label_id"]].dropna().drop_duplicates()
    return df, id_to_label

nli_df, id_to_label = normalize_labels(raw_df)
print(nli_df.shape)
print(nli_df["label_id"].value_counts().sort_index().rename(index=id_to_label))
nli_df.head()

## 5. Tokenization and DataLoaders

The custom model uses the same tokenizer as DistilBERT. This gives us a practical vocabulary and lets both approaches receive the same premise/hypothesis input format.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LENGTH = 96

if len(nli_df) > 2400:
    nli_df = nli_df.sample(2400, random_state=SEED).reset_index(drop=True)

stratify_labels = nli_df["label_id"] if nli_df["label_id"].nunique() > 1 and len(nli_df) >= 10 else None
train_df, val_df = train_test_split(
    nli_df,
    test_size=0.2,
    random_state=SEED,
    stratify=stratify_labels
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

In [ ]:
class NLIDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        encoded = self.tokenizer(
            str(row["premise"]),
            str(row["hypothesis"]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(row["label_id"], dtype=torch.long)
        return item

BATCH_SIZE = 16
train_loader = DataLoader(NLIDataset(train_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(NLIDataset(val_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE)

batch = next(iter(train_loader))
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

## 6. Custom Encoder Stack

The lightweight classifier below uses token embeddings, positional embeddings, multi-head self-attention blocks, feed-forward layers, masked mean pooling, and a classifier head.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(hidden_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, hidden_dim),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x, attention_mask=None, return_attention=False):
        attention_output, attention_weights = self.attention(
            x,
            attention_mask=attention_mask,
            return_attention=True
        )
        x = self.norm1(attention_output)
        x = self.norm2(x + self.feed_forward(x))

        if return_attention:
            return x, attention_weights
        return x


class CustomAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, num_labels, pad_token_id, max_length=128, hidden_dim=128, num_heads=4, ff_dim=256, num_layers=2, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_dim, padding_idx=pad_token_id)
        self.position_embedding = nn.Embedding(max_length, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            EncoderBlock(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self, input_ids, attention_mask=None, return_attentions=False):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.dropout(x)

        all_attentions = []
        for layer in self.layers:
            x, attention_weights = layer(x, attention_mask=attention_mask, return_attention=True)
            all_attentions.append(attention_weights)

        if attention_mask is None:
            pooled = x.mean(dim=1)
        else:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        logits = self.classifier(pooled)
        if return_attentions:
            return logits, all_attentions
        return logits


num_labels = max(3, nli_df["label_id"].nunique())
custom_model = CustomAttentionClassifier(
    vocab_size=tokenizer.vocab_size,
    num_labels=num_labels,
    pad_token_id=tokenizer.pad_token_id,
    max_length=MAX_LENGTH
).to(DEVICE)

with torch.no_grad():
    demo_batch = {key: value.to(DEVICE) for key, value in batch.items()}
    demo_logits, demo_attentions = custom_model(
        demo_batch["input_ids"],
        demo_batch["attention_mask"],
        return_attentions=True
    )

print("Logits shape:", demo_logits.shape)
print("Attention map shape from layer 0:", demo_attentions[0].shape)

## 7. Custom Model Training Loop

This is intentionally short. Increase `EPOCHS` and remove `MAX_TRAIN_BATCHES` for a more serious experiment.

In [ ]:
def train_one_epoch(model, loader, optimizer, max_batches=None):
    model.train()
    total_loss = 0.0
    steps = 0

    for batch_idx, batch in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    return total_loss / max(steps, 1)

def evaluate_model(model, loader, max_batches=None):
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            logits = model(input_ids, attention_mask)
            preds = logits.argmax(dim=-1)

            predictions.extend(preds.cpu().numpy().tolist())
            true_labels.extend(labels.cpu().numpy().tolist())

    return accuracy_score(true_labels, predictions), predictions, true_labels

optimizer = torch.optim.AdamW(custom_model.parameters(), lr=3e-4, weight_decay=0.01)
EPOCHS = 2
MAX_TRAIN_BATCHES = 40
MAX_VAL_BATCHES = 20

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(custom_model, train_loader, optimizer, max_batches=MAX_TRAIN_BATCHES)
    val_accuracy, _, _ = evaluate_model(custom_model, val_loader, max_batches=MAX_VAL_BATCHES)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_loss:.4f} - val_acc: {val_accuracy:.4f}")

## 8. DistilBERT Baseline

DistilBERT is a compact pretrained Transformer. It should usually outperform the small custom model because it already learned language representations from a massive corpus before fine-tuning.

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
).to(DEVICE)

bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)

def train_transformer_one_epoch(model, loader, optimizer, max_batches=40):
    model.train()
    total_loss = 0.0
    steps = 0

    for batch_idx, batch in enumerate(loader):
        if batch_idx >= max_batches:
            break

        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )
        outputs.loss.backward()
        optimizer.step()

        total_loss += outputs.loss.item()
        steps += 1

    return total_loss / max(steps, 1)

def evaluate_transformer(model, loader, max_batches=20):
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if batch_idx >= max_batches:
                break

            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            preds = outputs.logits.argmax(dim=-1)

            predictions.extend(preds.cpu().numpy().tolist())
            true_labels.extend(batch["labels"].cpu().numpy().tolist())

    return accuracy_score(true_labels, predictions), predictions, true_labels

bert_loss = train_transformer_one_epoch(bert_model, train_loader, bert_optimizer, max_batches=40)
bert_accuracy, bert_preds, bert_true = evaluate_transformer(bert_model, val_loader, max_batches=20)

print(f"DistilBERT short fine-tuning loss: {bert_loss:.4f}")
print(f"DistilBERT validation accuracy: {bert_accuracy:.4f}")

## 9. Attention Map Visualization

The custom model exposes attention weights directly. We visualize one head from one layer for a selected premise/hypothesis pair.

In [ ]:
def plot_custom_attention(model, tokenizer, premise, hypothesis, layer=0, head=0, max_length=96):
    model.eval()
    encoded = tokenizer(
        premise,
        hypothesis,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )
    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits, attentions = model(input_ids, attention_mask, return_attentions=True)

    valid_len = int(attention_mask[0].sum().item())
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0, :valid_len].cpu().tolist())
    attention_matrix = attentions[layer][0, head, :valid_len, :valid_len].cpu().numpy()

    plt.figure(figsize=(max(8, valid_len * 0.45), max(6, valid_len * 0.35)))
    sns.heatmap(attention_matrix, xticklabels=tokens, yticklabels=tokens, cmap="mako")
    plt.title(f"Custom attention map - layer {layer}, head {head}")
    plt.xticks(rotation=75)
    plt.yticks(rotation=0)
    plt.show()

    predicted_id = logits.argmax(dim=-1).item()
    print("Prediction:", id_to_label.get(predicted_id, predicted_id))


sample = val_df.iloc[0]
print("Premise:", sample["premise"])
print("Hypothesis:", sample["hypothesis"])
plot_custom_attention(custom_model, tokenizer, sample["premise"], sample["hypothesis"], layer=0, head=0, max_length=MAX_LENGTH)

## 10. Comparison and Reflection

| Criterion | Custom attention encoder | DistilBERT baseline |
|---|---|---|
| Training cost | Lightweight and fast for small experiments. | Heavier, but still efficient compared with full BERT. |
| Prior language knowledge | None beyond the dataset used here. It learns from scratch. | Strong pretrained language knowledge from large corpora. |
| Expected accuracy | Usually lower unless trained on a large dataset for longer. | Usually higher after even short fine-tuning. |
| Interpretability | Attention maps are easy to access because we control the implementation. | Attention can be inspected, but model internals are more complex. |
| Flexibility | Very flexible for learning, experimenting, and modifying architecture. | Best when the goal is strong practical performance quickly. |

**Insights about attention behavior**

The custom attention module shows how each token distributes focus over the sequence. In NLI, useful attention often appears around words that connect the premise and hypothesis: entities, negations, verbs, and semantic contrasts. For example, if the premise says someone is outdoors and the hypothesis says indoors, a good model should focus on those conflicting cues.

**Trade-off**

A lightweight attention stack is excellent for understanding the mechanics of Transformers and for controlled experiments. However, it starts from random weights, so it needs much more data and training to approach the performance of pretrained models. DistilBERT is less transparent and more expensive, but it brings strong language understanding immediately through transfer learning.

**Conclusion**

Single-head attention explains the core operation, multi-head attention makes the representation richer, and pretrained Transformers show why transfer learning is so powerful in modern NLP. For production NLI, a fine-tuned pretrained model is the better default. For learning and architecture experimentation, the custom implementation is more valuable because every part of the attention mechanism is visible and modifiable.